#  Modélisation Simple

**Objectif** : Créer un modèle pour prédire les maladies cardiaques

Nous allons suivre ces étapes :
1.  Préparer les données (nettoyage, encodage)
2.  Standardiser les variables
3.  Séparer les données (entraînement/test)
4.  Entraîner des modèles
5.  Évaluer les performances

In [1]:
# Importation des bibliothèques nécessaires
import pandas as pd  # Pour manipuler les données
import numpy as np   # Pour les calculs
import matplotlib.pyplot as plt  # Pour les graphiques
import seaborn as sns  # Pour des graphiques jolis

# Bibliothèques de machine learning
from sklearn.model_selection import train_test_split  # Pour séparer les données
from sklearn.preprocessing import StandardScaler, LabelEncoder  # Pour standardiser et encoder
from sklearn.linear_model import LogisticRegression  # Modèle de régression logistique
from sklearn.ensemble import RandomForestClassifier  # Modèle de forêt aléatoire
from sklearn.svm import SVC  # Modèle de machine à vecteurs de support

# Bibliothèques d'évaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score, roc_curve

print(" Bibliothèques importées avec succès !")

 Bibliothèques importées avec succès !


In [2]:
# Charger les données
df = pd.read_csv('../data/heart_disease_dataset.csv')

print(" Données chargées !")
print(f" {df.shape[0]} patients, {df.shape[1]} variables")

# Aperçu rapide
print("\n Aperçu des données :")
df.head()

 Données chargées !
 1000 patients, 16 variables

 Aperçu des données :


,Age,Gender,Cholesterol,Blood Pressure,Heart Rate,Smoking,Alcohol Intake,Exercise Hours,Family History,Diabetes,Obesity,Stress Level,Blood Sugar,Exercise Induced Angina,Chest Pain Type,Heart Disease
0,75,Female,228,119,66,Current,Heavy,1,No,No,Yes,8,119,Yes,Atypical Angina,1
1,48,Male,204,165,62,Current,NaN,5,No,No,No,9,70,Yes,Typical Angina,0
2,53,Male,234,91,67,Never,Heavy,3,Yes,No,Yes,5,196,Yes,Atypical Angina,1
3,69,Female,192,90,72,Current,NaN,4,No,Yes,No,7,107,Yes,Non-anginal Pain,0
4,62,Female,172,163,93,Never,NaN,6,No,Yes,No,2,183,Yes,Asymptomatic,0


In [3]:
# Étape 1: Préparation des données
print(" Préparation des données :")

# Vérifier les valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())

 Préparation des données :

Valeurs manquantes par colonne :
Age                          0
Gender                       0
Cholesterol                  0
Blood Pressure               0
Heart Rate                   0
Smoking                      0
Alcohol Intake             340
Exercise Hours               0
Family History               0
Diabetes                     0
Obesity                      0
Stress Level                 0
Blood Sugar                  0
Exercise Induced Angina      0
Chest Pain Type              0
Heart Disease                0
dtype: int64


In [5]:
plt.figure(figsize=(8,6))

sns.countplot(
    data=df,
    x='alcohol_intake',
    hue='alcohol_intake',
    palette='Set2',
    legend=False
)

plt.title("Distribution de Alcohol Intake")
plt.xlabel("Alcohol Intake")
plt.ylabel("Nombre")

plt.xticks(rotation=45)
plt.show()

ValueError: Could not interpret value `alcohol_intake` for `x`. An entry with this name does not appear in `data`.

<Figure size 800x600 with 0 Axes>

In [5]:
# Remplir les valeurs manquantes
# Pour les colonnes numériques : utiliser la médiane
# Pour les colonnes catégorielles : utiliser le mode

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col].fillna(df[col].median(), inplace=True)
            print(f"Remplissage de {col} avec la médiane")
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
            print(f"Remplissage de {col} avec le mode")

print("\n Valeurs manquantes après traitement :")
print(df.isnull().sum().sum())


 Valeurs manquantes après traitement :
0


In [ ]:
# Étape 2: Encoder les variables catégorielles
print(" Encodage des variables catégorielles :")

# Identifier les colonnes catégorielles
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Colonnes catégorielles à encoder : {list(categorical_cols)}")

# Encoder chaque colonne catégorielle
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"{col} encodée avec succès")

print("\n Types de données après encodage :")
print(df.dtypes)

In [ ]:
# Étape 3: Séparer les variables (features) et la cible (target)
print(" Séparation des variables :")

# X = toutes les variables sauf la cible
X = df.drop('Heart Disease', axis=1)  # axis=1 signifie qu'on supprime une colonne

# y = la variable cible (ce qu'on veut prédire)
y = df['Heart Disease']

print(f" Variables prédictives (X) : {X.shape[1]} variables")
print(f" Variable cible (y) : {y.name}")

print("\n Liste des variables prédictives :")
for i, col in enumerate(X.columns, 1):
    print(f"{i:2d}. {col}")

In [ ]:
# Étape 4: Séparer les données en ensemble d'entraînement et de test
print(" Séparation des données :")
print("- 80% pour entraîner le modèle")
print("- 20% pour tester le modèle")

# train_test_split divise automatiquement nos données
# random_state=42 assure que la division est toujours la même (reproductibilité)
# stratify=y garantit que la proportion de malades/non malades est la même dans les deux ensembles
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Données séparées :")
print(f" Entraînement : {X_train.shape[0]} patients ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f" Test : {X_test.shape[0]} patients ({X_test.shape[0]/len(X)*100:.1f}%)")

# Vérifier que les proportions sont bien conservées
print("\n Proportions de la variable cible :")
print(f"Entraînement - Malades : {y_train.mean()*100:.1f}%")
print(f"Test - Malades : {y_test.mean()*100:.1f}%")

In [ ]:
# Étape 5: Standardiser les variables numériques
print(" Standardisation des variables :")
print("La standardisation met toutes les variables à la même échelle")
print("(moyenne = 0, écart-type = 1)")

# Créer le standardiseur
scaler = StandardScaler()

# IMPORTANT : On apprend la standardisation SEULEMENT sur les données d'entraînement
# puis on applique la même transformation aux deux ensembles
X_train_scaled = scaler.fit_transform(X_train)  # fit_transform = apprend + transforme
X_test_scaled = scaler.transform(X_test)        # transform = applique seulement

print(" Standardisation terminée !")

# Convertir en DataFrame pour garder les noms de colonnes
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("\n Exemple de données standardisées (entraînement) :")
print(X_train_scaled.head().round(3))

In [ ]:
# Étape 6: Entraîner différents modèles
print(" Entraînement des modèles :")

# Dictionnaire pour stocker nos modèles
models = {
    'Régression Logistique': LogisticRegression(random_state=42, max_iter=1000),
    'Forêt Aléatoire': RandomForestClassifier(random_state=42, n_estimators=100),
    'Machine à Vecteurs de Support': SVC(random_state=42, probability=True)
}

# Dictionnaire pour stocker les prédictions
predictions = {}

# Entraîner chaque modèle et faire des prédictions
for name, model in models.items():
    print(f"\n Entraînement de {name}...")
    
    # Entraîner le modèle
    model.fit(X_train_scaled, y_train)
    
    # Faire des prédictions sur l'ensemble de test
    y_pred = model.predict(X_test_scaled)
    predictions[name] = y_pred
    
    # Calculer l'accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f" {name} - Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

In [ ]:
# Étape 7: Évaluer les performances de chaque modèle
print(" Évaluation détaillée des modèles :")

# Créer une figure pour les matrices de confusion
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Matrices de Confusion des Modèles', fontsize=16)

for i, (name, y_pred) in enumerate(predictions.items()):
    print(f"\n {name} :")
    
    # Rapport de classification
    report = classification_report(y_test, y_pred, target_names=['Non Malade', 'Malade'])
    print(report)
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    
    # Visualisation
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Non Malade', 'Malade'],
                yticklabels=['Non Malade', 'Malade'])
    axes[i].set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.3f}')
    axes[i].set_xlabel('Prédiction')
    axes[i].set_ylabel('Réalité')

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des performances avec un graphique
print(" Comparaison des performances :")

# Calculer les métriques pour chaque modèle
metrics = []
for name, y_pred in predictions.items():
    accuracy = accuracy_score(y_test, y_pred)
    
    # Calculer AUC (Area Under Curve)
    model = models[name]
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_prob = model.decision_function(X_test_scaled)
    
    auc = roc_auc_score(y_test, y_prob)
    
    metrics.append({
        'Modèle': name,
        'Accuracy': accuracy,
        'AUC': auc
    })

# Créer un DataFrame pour faciliter la visualisation
metrics_df = pd.DataFrame(metrics)

# Graphique de comparaison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Accuracy
sns.barplot(data=metrics_df, x='Modèle', y='Accuracy', ax=ax1, palette='viridis')
ax1.set_title('Accuracy des Modèles')
ax1.set_ylim(0, 1)
for i, v in enumerate(metrics_df['Accuracy']):
    ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

# AUC
sns.barplot(data=metrics_df, x='Modèle', y='AUC', ax=ax2, palette='plasma')
ax2.set_title('AUC des Modèles')
ax2.set_ylim(0, 1)
for i, v in enumerate(metrics_df['AUC']):
    ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n Tableau récapitulatif :")
print(metrics_df.round(3))

In [ ]:
# Trouver le meilleur modèle
print(" Détermination du meilleur modèle :")

# Basé sur l'accuracy
best_accuracy = metrics_df.loc[metrics_df['Accuracy'].idxmax()]
print(f"\n Meilleur accuracy : {best_accuracy['Modèle']} ({best_accuracy['Accuracy']:.3f})")

# Basé sur l'AUC
best_auc = metrics_df.loc[metrics_df['AUC'].idxmax()]
print(f" Meilleur AUC : {best_auc['Modèle']} ({best_auc['AUC']:.3f})")

# Choisir le meilleur modèle globalement (moyenne des deux métriques)
metrics_df['Score_Combiné'] = (metrics_df['Accuracy'] + metrics_df['AUC']) / 2
best_overall = metrics_df.loc[metrics_df['Score_Combiné'].idxmax()]
print(f"\n Meilleur modèle global : {best_overall['Modèle']}")
print(f"Score combiné : {best_overall['Score_Combiné']:.3f}")

In [ ]:
# Tester le meilleur modèle sur de nouvelles données (exemple)
print(" Test du meilleur modèle sur un exemple :")

# Prendre le meilleur modèle
best_model_name = best_overall['Modèle']
best_model = models[best_model_name]

# Prendre un patient de l'ensemble de test comme exemple
example_patient = X_test_scaled.iloc[0:1]  # Premier patient
true_label = y_test.iloc[0]

print(f" Patient exemple :")
print(f"Vraie étiquette : {'Malade' if true_label == 1 else 'Non malade'}")

# Faire une prédiction
prediction = best_model.predict(example_patient)[0]
prediction_proba = best_model.predict_proba(example_patient)[0]

print(f"\n Prédiction du modèle ({best_model_name}) :")
print(f"Résultat : {'Malade' if prediction == 1 else 'Non malade'}")
print(f"Probabilités :")
print(f"  - Non malade : {prediction_proba[0]:.3f} ({prediction_proba[0]*100:.1f}%)")
print(f"  - Malade : {prediction_proba[1]:.3f} ({prediction_proba[1]*100:.1f}%)")

# Vérifier si la prédiction est correcte
if prediction == true_label:
    print("\n Prédiction CORRECTE !")
else:
    print("\n Prédiction INCORRECTE")

##  Résumé de notre Modélisation

### Ce que nous avons accompli :

1. ** Préparation des données** :
   - Remplissage des valeurs manquantes
   - Encodage des variables catégorielles
   - Séparation variables/cible
   - Division entraînement/test (80%/20%)
   - Standardisation des variables

2. ** Modèles entraînés** :
   - Régression Logistique
   - Forêt Aléatoire
   - Machine à Vecteurs de Support

3. ** Évaluation des performances** :
   - Accuracy et AUC pour chaque modèle
   - Matrices de confusion
   - Rapports de classification détaillés

4. ** Meilleur modèle identifié** :
   - Basé sur les métriques combinées
   - Test sur un exemple concret

### Interprétation médicale :

- **Accuracy** : Pourcentage de prédictions correctes globales
- **Precision** : Fiabilité des prédictions positives
- **Recall** : Capacité à détecter tous les malades
- **F1-score** : Équilibre entre precision et recall

### Prochaines étapes possibles :
1.  Optimiser les hyperparamètres
2.  Essayer des modèles plus avancés
3.  Analyser l'importance des variables
4.  Sauvegarder le modèle pour déploiement

**Félicitations ! Vous avez créé votre premier pipeline de machine learning !** 